In [ ]:
!pip install torch torchvision transformers datasets scikit-learn tqdm modelscope addict remotezip sentencepiece protobuf pillow --break-system-packages

In [17]:
# Mount Drive so checkpoints/outputs persist across Colab sessions.
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
import io
import random
import numpy as np
from PIL import Image, ImageFilter
from torchvision.transforms import ColorJitter

# Exact transform pool from the hackathon spec — one is applied per image, at random
def apply_robustness_transform(image: Image.Image) -> Image.Image:

    transform_name = random.choice(["jpeg", "blur", "resize", "noise", "color", "crop"])

    if transform_name == "jpeg":
        quality = random.choice([90, 70, 50, 30])
        buffer = io.BytesIO()
        image.save(buffer, format="JPEG", quality=quality)
        buffer.seek(0)
        return Image.open(buffer).convert("RGB")

    elif transform_name == "blur":
        sigma = random.choice([0.5, 1.0, 2.0])
        return image.filter(ImageFilter.GaussianBlur(radius=sigma))

    elif transform_name == "resize":
        scale = random.choice([0.5, 0.25])
        w, h = image.size
        small = image.resize((max(1, int(w * scale)), max(1, int(h * scale))), Image.BILINEAR)
        return small.resize((w, h), Image.BILINEAR)

    elif transform_name == "noise":
        sigma = random.choice([0.02, 0.05, 0.10])
        arr = np.array(image).astype(np.float32) / 255.0
        noise = np.random.normal(0, sigma, arr.shape)
        arr = np.clip(arr + noise, 0, 1) * 255.0
        return Image.fromarray(arr.astype(np.uint8))

    elif transform_name == "color":
        jitter = ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2)
        return jitter(image)

    elif transform_name == "crop":
        w, h = image.size
        cw, ch = int(w * 0.8), int(h * 0.8)
        x0, y0 = (w - cw) // 2, (h - ch) // 2
        cropped = image.crop((x0, y0, x0 + cw, y0 + ch))
        return cropped.resize((w, h), Image.BILINEAR)

    return image

In [19]:
import os
import csv
import random
import torch
import torch.nn as nn
from torch.utils.data import IterableDataset, Dataset, DataLoader
from datasets import load_dataset
from transformers import AutoModel, AutoProcessor, AutoConfig
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from tqdm.auto import tqdm
from PIL import Image

# ============================================================
# Configuration
# ============================================================

MODEL_NAME = "google/siglip-base-patch16-224"
DATASET_NAME = "saberzl/SID_Set"
BATCH_SIZE = 16
EPOCHS = 4
LEARNING_RATE = 1e-4
SEED = 42
OUTPUT_DIR = "/content/drive/MyDrive/TechJam/checkpoints"
FREEZE_BACKBONE = True

# --- Streaming settings ---
SHUFFLE_BUFFER = 1000

# Batches of real training exposure per epoch.
# # of images seen = # batches * # batch_size * # epochs
MAX_TRAIN_STEPS_PER_EPOCH = 800

# Batches used for the quick per-epoch progress check (not the robustness/
# error-analysis cells below, which use the fixed 300-image sample instead).
MAX_EVAL_STEPS = 150

# Training-time robustness augmentation.
AUGMENT_PROB = 0.7

# --- WildFake mix-in (see pull_wildfake_balanced.py) ---
# Adds cross-generator diversity beyond SID_Set.
WILDFAKE_TRAIN_RATIO = 0.15   # fraction of each training batch drawn from WildFake
WILDFAKE_DIR = "wildfake_mix"

# ============================================================
# Label Mapping & Data Handling
# ============================================================

LABEL_MAP = {
    "real": 0.0,
    "full_synthetic": 1.0,
    "tampered": 1.0
}

def get_label(example):
    """
    Map a raw SID_Set example to a float target:
    0.0 = real, 1.0 = AI-generated or edited.
    """
    for key in ("label", "category", "class", "type"):
        if key not in example:
            continue
        value = example[key]
        if isinstance(value, int):
            return 0.0 if value == 0 else 1.0
        if isinstance(value, str) and value.lower() in LABEL_MAP:
            return LABEL_MAP[value.lower()]

    raise KeyError(
        f"Could not find a usable label. Available keys: {list(example.keys())}"
    )

class HFIterableDataset(IterableDataset):
    """Wraps a Hugging Face streaming dataset to serve converted images and
    labels, optionally applying the hackathon's robustness-transform pool
    (train-time augmentation only — leave augment=False for validation)."""
    def __init__(self, hf_dataset, augment=False, augment_prob=0.7):
        self.hf_dataset = hf_dataset
        self.augment = augment
        self.augment_prob = augment_prob

    def __iter__(self):
        for item in self.hf_dataset:
            image = item["image"]
            if image is None:
                continue
            if image.mode != "RGB":
                image = image.convert("RGB")
            if self.augment and random.random() < self.augment_prob:
                image = apply_robustness_transform(image)
            label = get_label(item)
            yield {"image": image, "label": label}

class LocalLabeledImageDataset(Dataset):
    """Map-style dataset reading a labels.csv produced by
    pull_wildfake_balanced.py (columns: image_path,category,generator,label),
    relative to `root`. Supports the same optional augmentation policy as
    HFIterableDataset so both data sources can be trained identically."""
    def __init__(self, root, augment=False, augment_prob=0.7):
        self.root = root
        self.augment = augment
        self.augment_prob = augment_prob
        self.rows = []
        labels_path = os.path.join(root, "labels.csv")
        if os.path.exists(labels_path):
            with open(labels_path, newline="") as f:
                for row in csv.DictReader(f):
                    self.rows.append((row["image_path"], float(row["label"])))

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        rel_path, label = self.rows[idx]
        image = Image.open(os.path.join(self.root, rel_path)).convert("RGB")
        if self.augment and random.random() < self.augment_prob:
            image = apply_robustness_transform(image)
        return {"image": image, "label": label}

class MixedIterableDataset(IterableDataset):
    """Interleaves an HF streaming dataset with draws from a small local
    pool (e.g. WildFake): with probability `ratio` per yielded example, a
    local example replaces the next HF example. The local pool is looped
    and reshuffled once exhausted, since it's much smaller than the number
    of training steps."""
    def __init__(self, hf_iterable, local_dataset, ratio=0.15, seed=42):
        self.hf_iterable = hf_iterable
        self.local_dataset = local_dataset
        self.ratio = ratio
        self.seed = seed

    def _local_cycle(self):
        rng = random.Random(self.seed)
        indices = list(range(len(self.local_dataset)))
        while True:
            rng.shuffle(indices)
            for i in indices:
                yield self.local_dataset[i]

    def __iter__(self):
        local_iter = self._local_cycle() if len(self.local_dataset) > 0 else None
        for item in self.hf_iterable:
            if local_iter is not None and random.random() < self.ratio:
                yield next(local_iter)
            else:
                yield item

def collate_fn_factory(processor):
    """Preprocesses a batch using the SigLIP AutoProcessor."""
    def collate_fn(batch):
        images = [item["image"] for item in batch]
        labels = torch.tensor([item["label"] for item in batch], dtype=torch.float32)
        try:
            pixel_values = processor(images=images, return_tensors="pt")["pixel_values"]
            return pixel_values, labels
        except Exception as e:
            print(f"Skipping batch due to processing error: {e}")
            return None, None
    return collate_fn

# ============================================================
# Model Architecture
# ============================================================

class VisionConfidenceScorer(nn.Module):
    """Wraps a CLIP/SigLIP vision tower + a single-logit head for a confidence score."""
    def __init__(self, backbone_name, freeze_backbone=True):
        super().__init__()
        config = AutoConfig.from_pretrained(backbone_name)
        full_model = AutoModel.from_pretrained(backbone_name)

        # Pull the vision tower exclusively
        self.vision_model = full_model.vision_model
        hidden_size = config.vision_config.hidden_size

        if freeze_backbone:
            for p in self.vision_model.parameters():
                p.requires_grad = False

        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_size // 2, 1)  # Outputs raw logit
        )

    def forward(self, pixel_values):
        outputs = self.vision_model(pixel_values=pixel_values)
        return self.head(outputs.pooler_output).squeeze(-1)

    @torch.no_grad()
    def confidence(self, pixel_values):
        """Returns binary probability in range [0, 1]"""
        logit = self.forward(pixel_values)
        return torch.sigmoid(logit)

# ============================================================
# Evaluation Logic
# ============================================================

@torch.no_grad()
def evaluate(model, loader, device, eval_steps=None):
    """
    Evaluates the model over the validation loader stream.
    Calculates AUROC, Average Precision (AP), Accuracy at 0.5, and Youden's J.
    """
    model.eval()
    all_scores, all_labels = [], []

    eval_pbar = tqdm(loader, desc="Evaluating", total=eval_steps, leave=False)
    for i, batch in enumerate(eval_pbar):
        pixel_values, labels = batch
        if pixel_values is None:
            continue

        pixel_values = pixel_values.to(device)
        scores = model.confidence(pixel_values).cpu()
        all_scores.extend(scores.tolist())
        all_labels.extend(labels.tolist())

        if eval_steps is not None and i + 1 >= eval_steps:
            break

    if not all_labels:
        print("Warning: No data was evaluated.")
        return

    auroc = roc_auc_score(all_labels, all_scores)
    ap = average_precision_score(all_labels, all_scores)
    print(f"\nEvaluation Results:")
    print(f"  AUROC:             {auroc:.4f}")
    print(f"  Average Precision: {ap:.4f}")

    threshold = 0.5
    preds = [1.0 if s >= threshold else 0.0 for s in all_scores]
    acc = sum(p == l for p, l in zip(preds, all_labels)) / len(all_labels)
    print(f"  Accuracy @ 0.5:    {acc:.4f}")

    fpr, tpr, thresholds = roc_curve(all_labels, all_scores)
    best_idx = (tpr - fpr).argmax()
    print(f"  Suggested threshold (Youden's J): {thresholds[best_idx]:.4f}\n")

In [20]:
# ============================================================
# Execution Setup
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

processor = AutoProcessor.from_pretrained(MODEL_NAME)
collate_fn = collate_fn_factory(processor)

# --- Pull the WildFake mix-in (train + eval pools), if not already present ---
wildfake_train_dir = os.path.join(WILDFAKE_DIR, "train")
wildfake_eval_dir = os.path.join(WILDFAKE_DIR, "eval")
if not os.path.exists(os.path.join(wildfake_train_dir, "labels.csv")):
    print("Pulling WildFake train/eval mix (pull_wildfake_balanced.py)...")
    !python "/content/drive/MyDrive/TechJam/pull_wildfake_balanced.py"
else:
    print(f"Found existing WildFake mix at {WILDFAKE_DIR}, skipping pull.")

# --- Training Loader ---
print("Loading training data stream...")
train_stream = load_dataset(DATASET_NAME, split="train", streaming=True)
shuffled_train_stream = train_stream.shuffle(buffer_size=SHUFFLE_BUFFER, seed=SEED)
train_dataset_sid = HFIterableDataset(shuffled_train_stream, augment=True, augment_prob=AUGMENT_PROB)

wildfake_train_local = LocalLabeledImageDataset(wildfake_train_dir, augment=True, augment_prob=AUGMENT_PROB)
print(f"WildFake train pool: {len(wildfake_train_local)} images "
      f"(mixed in at ~{WILDFAKE_TRAIN_RATIO:.0%} of training examples)")
train_dataset = MixedIterableDataset(train_dataset_sid, wildfake_train_local, ratio=WILDFAKE_TRAIN_RATIO, seed=SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    collate_fn=collate_fn,
    num_workers=0
)

# --- Validation Loader (streaming, used for the per-epoch training-progress eval) ---
print("Loading validation data stream...")
val_stream = load_dataset(DATASET_NAME, split="validation", streaming=True)
val_dataset = HFIterableDataset(val_stream)  # augment=False: clean eval
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    collate_fn=collate_fn,
    num_workers=0
)

# --- Fixed clean validation sample ---
# Materialized once (as PIL images, not tensors) so the robustness evaluation
# and error analysis cells later can score the same images clean and under
# every transform, instead of comparing against a different random streaming
# slice each time.
N_FIXED_EVAL = 300
print(f"Caching a fixed sample of {N_FIXED_EVAL} clean SID_Set validation images...")
fixed_val_stream = load_dataset(DATASET_NAME, split="validation", streaming=True)
fixed_eval_examples = []
for item in fixed_val_stream:
    image = item["image"]
    if image is None:
        continue
    if image.mode != "RGB":
        image = image.convert("RGB")
    label = get_label(item)
    fixed_eval_examples.append((image, label, "sid_set"))
    if len(fixed_eval_examples) >= N_FIXED_EVAL:
        break
print(f"Cached {len(fixed_eval_examples)} SID_Set examples "
      f"({sum(1 for _, l, _ in fixed_eval_examples if l == 0.0)} real / "
      f"{sum(1 for _, l, _ in fixed_eval_examples if l == 1.0)} AIGC).")

wildfake_eval_local = LocalLabeledImageDataset(wildfake_eval_dir, augment=False)
for i in range(len(wildfake_eval_local)):
    item = wildfake_eval_local[i]
    fixed_eval_examples.append((item["image"], item["label"], "wildfake"))
print(f"Added {len(wildfake_eval_local)} WildFake eval examples "
      f"({sum(1 for _, l, s in fixed_eval_examples if s == 'wildfake' and l == 0.0)} real / "
      f"{sum(1 for _, l, s in fixed_eval_examples if s == 'wildfake' and l == 1.0)} AIGC). "
      f"Fixed eval sample now {len(fixed_eval_examples)} images total.")

# Initialize Model, Optimizer, and Loss
model = VisionConfidenceScorer(MODEL_NAME, freeze_backbone=FREEZE_BACKBONE).to(device)
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=0.01)
criterion = nn.BCEWithLogitsLoss()
n_trainable = sum(p.numel() for p in trainable_params)
n_total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {n_trainable:,} / {n_total:,} total "
      f"(backbone frozen: {FREEZE_BACKBONE})")

# ============================================================
# Training & Evaluation Loop
# ============================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)
global_step = 0

print("Starting training script...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    epoch_batches = 0

    pbar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}",
        total=MAX_TRAIN_STEPS_PER_EPOCH
    )

    for i, batch in enumerate(pbar):
        pixel_values, labels = batch
        if pixel_values is None:
            continue

        pixel_values = pixel_values.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(pixel_values)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        global_step += 1
        epoch_batches += 1
        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

        # Stop epoch training early if batch step count limit is reached
        if MAX_TRAIN_STEPS_PER_EPOCH is not None and epoch_batches >= MAX_TRAIN_STEPS_PER_EPOCH:
            break

    avg_loss = total_loss / epoch_batches if epoch_batches > 0 else 0
    print(f"\nEpoch {epoch+1} average training loss: {avg_loss:.4f}")

    # --- Run Evaluation ---
    print(f"Running evaluation for Epoch {epoch+1}...")
    evaluate(model, val_loader, device, eval_steps=MAX_EVAL_STEPS)

    # Save checkpoint
    ckpt_path = os.path.join(OUTPUT_DIR, f"classifier_epoch{epoch+1}.pt")
    torch.save(model.state_dict(), ckpt_path)
    print(f"Saved checkpoint to {ckpt_path}\n")

print("\nTraining complete.")

Using device: cuda
Pulling WildFake train/eval mix (pull_wildfake_balanced.py)...
Sampling 40 train + 20 eval per category (6 real + 6 fake categories):
  afhq           label=0  train=40 eval=20 / 31933 available
  celebahq       label=0  train=40 eval=20 / 30000 available
  church         label=0  train=40 eval=20 / 83352 available
  ffhq           label=0  train=40 eval=20 / 70000 available
  imagenet       label=0  train=40 eval=20 / 96788 available
  laion5b        label=0  train=40 eval=20 / 271831 available
  DDIM           label=1  train=40 eval=20 / 65713 available
  DDPM           label=1  train=40 eval=20 / 76561 available
  Imagen         label=1  train=40 eval=20 / 47435 available
  VQDM           label=1  train=40 eval=20 / 153479 available
  ADM            label=1  train=40 eval=20 / 155022 available
  SDwithAdaptor  label=1  train=40 eval=20 / 199982 available

train: 480 images -> real(0)=240  fake(1)=240  (wildfake_mix/train/labels.csv)
eval: 240 images -> real(0)=120

Resolving data files:   0%|          | 0/249 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/34 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/249 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/34 [00:00<?, ?it/s]

...


Evaluation Results:
  AUROC:             0.9419
  Average Precision: 0.9718
  Accuracy @ 0.5:    0.8788
  Suggested threshold (Youden's J): 0.6690

Saved checkpoint to /content/drive/MyDrive/TechJam/checkpoints/classifier_epoch3.pt



Epoch 4/4:   0%|          | 0/800 [00:00<?, ?it/s]


Epoch 4 average training loss: 0.2334
Running evaluation for Epoch 4...


Evaluating:   0%|          | 0/150 [00:00<?, ?it/s]


Evaluation Results:
  AUROC:             0.9448
  Average Precision: 0.9731
  Accuracy @ 0.5:    0.8804
  Suggested threshold (Youden's J): 0.6511

Saved checkpoint to /content/drive/MyDrive/TechJam/checkpoints/classifier_epoch4.pt


Training complete.


In [21]:
import json

# ============================================================
# Directory Scorer (Colab convenience wrapper)
# ============================================================
# This is the deliverable-shaped scorer: takes an image directory, outputs a
# JSON list of {"image_path", "pred"}.

def score_directory_colab(input_dir, output_json_path, checkpoint_path, backbone_name="google/siglip-base-patch16-224"):
    # Detect GPU
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")
    if device == "cpu":
         print("⚠️ WARNING: Running on CPU. It is highly recommended to change your Colab runtime to GPU (Runtime -> Change runtime type -> T4 GPU)!")

    # Load processor and initialize model
    print("Loading model architecture and processor...")
    processor = AutoProcessor.from_pretrained(backbone_name)
    model = VisionConfidenceScorer(backbone_name)

    # Load trained weights
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"❌ Checkpoint not found at: {checkpoint_path}. Please check your checkpoint path!")

    print(f"Loading weights from {checkpoint_path}...")
    model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
    model.to(device)
    model.eval()

    # Verify input directory exists
    if not os.path.exists(input_dir):
        # Create a dummy folder so the code doesn't immediately crash if the user hasn't uploaded images yet
        os.makedirs(input_dir, exist_ok=True)
        print(f"Created a new empty folder at: {input_dir}. Please upload your images here in the Colab file tree!")
        return

    # Find all images in the directory
    valid_extensions = ('.png', '.jpg', '.jpeg', '.webp', '.bmp')
    image_files = [
        f for f in os.listdir(input_dir)
        if f.lower().endswith(valid_extensions)
    ]

    if not image_files:
        print(f"No valid images found in folder: {input_dir}")
        print("Please drag and drop some test images (.png, .jpg, etc.) into the Colab file explorer on the left.")
        return

    print(f"Found {len(image_files)} test images. Starting inference...")
    results = []

    # Process each image
    for filename in tqdm(image_files, desc="Scoring Images"):
        file_path = os.path.join(input_dir, filename)

        try:
            # 1. Load and prepare image (scored as-is — no transform applied)
            image = Image.open(file_path).convert("RGB")

            # 2. Preprocess
            pixel_values = processor(images=[image], return_tensors="pt")["pixel_values"].to(device)

            # 3. Predict
            score = model.confidence(pixel_values).item()

            # 4. Store result
            results.append({
                "image_path": file_path,
                "pred": round(score, 4)
            })

        except Exception as e:
            print(f"\nSkipped {filename} due to error: {e}")

    # Save to JSON
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=4)

    print(f"\nDone! Saved prediction outputs to {output_json_path}")
    print(f"You can find your JSON in Colab's file tree on the left side menu!")

# ============================================================
# Colab Directory Paths Configuration
# ============================================================

# Colab uses '/content' as its root directory.
# Adjust these paths depending on where your model files are!
INPUT_IMAGE_DIR = "/content/drive/MyDrive/TechJam/my_test_images"               # Place your test images here
OUTPUT_JSON_FILE = "/content/drive/MyDrive/TechJam/aigc_predictions.json"       # Output predictions path
CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, f"classifier_epoch{EPOCHS}.pt")      # Latest checkpoint from the training run above

# Run the Colab scoring pipeline
score_directory_colab(
    input_dir=INPUT_IMAGE_DIR,
    output_json_path=OUTPUT_JSON_FILE,
    checkpoint_path=CHECKPOINT_FILE
)


🤖 Using device: cuda
📥 Loading model architecture and processor...


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

⚙️ Loading weights from /content/drive/MyDrive/TechJam/checkpoints/classifier_epoch4.pt...
🔍 Found 8 test images. Starting inference...


Scoring Images:   0%|          | 0/8 [00:00<?, ?it/s]


🎉 Done! Saved prediction outputs to /content/drive/MyDrive/TechJam/aigc_predictions.json
You can find your JSON in Colab's file tree on the left side menu!


# Robustness Evaluation (required deliverable)

Scores the same fixed 300-image validation sample (`fixed_eval_examples`, cached
above) clean, and again under every transform/severity in the hackathon's spec —
deterministically (fixed severities, not the random single-draw used for training
augmentation) so the comparison is apples-to-apples. Produces the clean-vs-transformed
table the submission's "Robustness Evaluation Summary" needs.

In [22]:
import pandas as pd

# ============================================================
# Deterministic transform pool (one fixed severity applied per call — as
# opposed to apply_robustness_transform, which is random and used only for
# training-time augmentation).
# ============================================================

def apply_fixed_transform(image: Image.Image, name: str, severity) -> Image.Image:
    if name == "jpeg":
        buffer = io.BytesIO()
        image.save(buffer, format="JPEG", quality=severity)
        buffer.seek(0)
        return Image.open(buffer).convert("RGB")
    elif name == "blur":
        return image.filter(ImageFilter.GaussianBlur(radius=severity))
    elif name == "resize":
        w, h = image.size
        small = image.resize((max(1, int(w * severity)), max(1, int(h * severity))), Image.BILINEAR)
        return small.resize((w, h), Image.BILINEAR)
    elif name == "noise":
        arr = np.array(image).astype(np.float32) / 255.0
        rng = np.random.RandomState(0)  # fixed seed so results are reproducible
        noise = rng.normal(0, severity, arr.shape)
        arr = np.clip(arr + noise, 0, 1) * 255.0
        return Image.fromarray(arr.astype(np.uint8))
    elif name == "color":
        jitter = ColorJitter(brightness=(severity, severity), contrast=(severity, severity), saturation=(severity, severity))
        return jitter(image)
    elif name == "crop":
        w, h = image.size
        cw, ch = int(w * severity), int(h * severity)
        x0, y0 = (w - cw) // 2, (h - ch) // 2
        cropped = image.crop((x0, y0, x0 + cw, y0 + ch))
        return cropped.resize((w, h), Image.BILINEAR)
    raise ValueError(f"Unknown transform: {name}")

# Transform pool + severities, matching the hackathon spec table exactly.
TRANSFORM_SPEC = {
    "jpeg":   [90, 70, 50, 30],
    "blur":   [0.5, 1.0, 2.0],
    "resize": [0.5, 0.25],
    "noise":  [0.02, 0.05, 0.10],
    "color":  [1.2],   # brightness/contrast/saturation +20%
    "crop":   [0.8],   # center crop 80%
}

@torch.no_grad()
def score_examples(model, processor, examples, device, transform_fn=None, batch_size=16):
    """Scores a list of (PIL image, label, source) triples, optionally after transform_fn."""
    model.eval()
    scores, labels = [], []
    for i in range(0, len(examples), batch_size):
        batch = examples[i:i + batch_size]
        images = [transform_fn(img) if transform_fn else img for img, _, _ in batch]
        labels.extend(lbl for _, lbl, _ in batch)
        pixel_values = processor(images=images, return_tensors="pt")["pixel_values"].to(device)
        scores.extend(model.confidence(pixel_values).cpu().tolist())
    return scores, labels

def summarize(scores, labels, threshold=0.5):
    auroc = roc_auc_score(labels, scores)
    ap = average_precision_score(labels, scores)
    preds = [1.0 if s >= threshold else 0.0 for s in scores]
    acc = sum(p == l for p, l in zip(preds, labels)) / len(labels)
    return {"AUROC": auroc, "AP": ap, "Accuracy@0.5": acc}

# --- Clean baseline (full mixed sample: SID_Set + WildFake) ---
print("Scoring clean baseline...")
clean_scores, clean_labels = score_examples(model, processor, fixed_eval_examples, device)
robustness_rows = [{"transform": "clean", "severity": "-", **summarize(clean_scores, clean_labels)}]

# --- Each transform x severity (also on the full mixed sample) ---
for name, severities in TRANSFORM_SPEC.items():
    for severity in severities:
        print(f"Scoring transform={name} severity={severity}...")
        t_scores, t_labels = score_examples(
            model, processor, fixed_eval_examples, device,
            transform_fn=lambda img, n=name, s=severity: apply_fixed_transform(img, n, s)
        )
        robustness_rows.append({"transform": name, "severity": severity, **summarize(t_scores, t_labels)})

robustness_df = pd.DataFrame(robustness_rows)
robustness_df["AUROC_drop_vs_clean"] = robustness_df["AUROC"].iloc[0] - robustness_df["AUROC"]
print("\nRobustness Evaluation Summary:")
display(robustness_df.round(4))

robustness_df.to_csv(os.path.join(OUTPUT_DIR, "robustness_summary.csv"), index=False)
print(f"\nSaved to {os.path.join(OUTPUT_DIR, 'robustness_summary.csv')} — pull this into the DevPost writeup / repo README.")


sources = [src for _, _, src in fixed_eval_examples]
by_source_rows = []
for src in sorted(set(sources)):
    idx = [i for i, s in enumerate(sources) if s == src]
    src_scores = [clean_scores[i] for i in idx]
    src_labels = [clean_labels[i] for i in idx]
    by_source_rows.append({"source": src, "n": len(idx), **summarize(src_scores, src_labels)})

by_source_df = pd.DataFrame(by_source_rows)
print("\nClean-image performance by data source (cross-dataset generalization check):")
display(by_source_df.round(4))
by_source_df.to_csv(os.path.join(OUTPUT_DIR, "by_source_summary.csv"), index=False)

Scoring clean baseline...
Scoring transform=jpeg severity=90...
Scoring transform=jpeg severity=70...
Scoring transform=jpeg severity=50...
Scoring transform=jpeg severity=30...
Scoring transform=blur severity=0.5...
Scoring transform=blur severity=1.0...
Scoring transform=blur severity=2.0...
Scoring transform=resize severity=0.5...
Scoring transform=resize severity=0.25...
Scoring transform=noise severity=0.02...
Scoring transform=noise severity=0.05...
Scoring transform=noise severity=0.1...
Scoring transform=color severity=1.2...
Scoring transform=crop severity=0.8...

Robustness Evaluation Summary:


,transform,severity,AUROC,AP,Accuracy@0.5,AUROC_drop_vs_clean
0,clean,-,0.9658,0.9775,0.8944,0.0000
1,jpeg,90,0.9759,0.9841,0.9167,-0.0102
2,jpeg,70,0.9783,0.9858,0.9167,-0.0126
3,jpeg,50,0.9762,0.9841,0.9167,-0.0104
4,jpeg,30,0.9742,0.9828,0.9074,-0.0085
5,blur,0.5,0.9640,0.9761,0.8796,0.0017
6,blur,1.0,0.9610,0.9750,0.8870,0.0047
7,blur,2.0,0.9662,0.9778,0.9000,-0.0005
8,resize,0.5,0.9620,0.9754,0.8944,0.0037
9,resize,0.25,0.9663,0.9775,0.8926,-0.0005



Saved to /content/drive/MyDrive/TechJam/checkpoints/robustness_summary.csv — pull this into the DevPost writeup / repo README.

Clean-image performance by data source (cross-dataset generalization check):


,source,n,AUROC,AP,Accuracy@0.5
0,sid_set,300,0.9582,0.9798,0.89
1,wildfake,240,0.9749,0.9760,0.90


# Error Analysis (required deliverable)

Pulls the most confident false positives (real images scored high) and false
negatives (AIGC images scored low) from the clean-baseline pass above, saves
thumbnails + scores so they can be dropped straight into the DevPost error
analysis note.

In [23]:
ERROR_ANALYSIS_DIR = os.path.join(OUTPUT_DIR, "error_analysis")
os.makedirs(ERROR_ANALYSIS_DIR, exist_ok=True)
TOP_K = 8

# (index, image, label, source, score) for every misclassified example at threshold 0.5
scored = [(i, img, lbl, src, s) for i, ((img, lbl, src), s) in enumerate(zip(fixed_eval_examples, clean_scores))]
false_positives = sorted(
    [row for row in scored if row[2] == 0.0 and row[4] >= 0.5],
    key=lambda x: -x[4]
)[:TOP_K]
false_negatives = sorted(
    [row for row in scored if row[2] == 1.0 and row[4] < 0.5],
    key=lambda x: x[4]
)[:TOP_K]

print(f"False positives (real, scored as AIGC): {len(false_positives)} shown / "
      f"{sum(1 for row in scored if row[2] == 0.0 and row[4] >= 0.5)} total in sample")
print(f"False negatives (AIGC, scored as real): {len(false_negatives)} shown / "
      f"{sum(1 for row in scored if row[2] == 1.0 and row[4] < 0.5)} total in sample\n")

def save_error_set(name, items):
    rows = []
    for rank, (idx, img, lbl, src, score) in enumerate(items):
        fname = f"{name}_{rank:02d}_score{score:.3f}.png"
        img.save(os.path.join(ERROR_ANALYSIS_DIR, fname))
        rows.append({"rank": rank, "fixed_eval_index": idx, "label": lbl, "source": src, "score": round(score, 4), "file": fname})
    return pd.DataFrame(rows)

fp_df = save_error_set("false_positive", false_positives)
fn_df = save_error_set("false_negative", false_negatives)

print("Top false positives (real -> scored as AIGC):")
display(fp_df)
print("\nTop false negatives (AIGC -> scored as real):")
display(fn_df)

pd.concat([fp_df.assign(type="false_positive"), fn_df.assign(type="false_negative")]).to_csv(
    os.path.join(ERROR_ANALYSIS_DIR, "error_analysis.csv"), index=False
)
print(f"\nThumbnails + CSV saved to {ERROR_ANALYSIS_DIR} — inspect these and write 2-3 "
      f"sentences per failure mode for the DevPost error analysis note (e.g. is there a "
      f"visible pattern — heavy compression, faces, particular textures, or a specific "
      f"WildFake generator underperforming vs. SID_Set?).")

False positives (real, scored as AIGC): 8 shown / 25 total in sample
False negatives (AIGC, scored as real): 8 shown / 32 total in sample

Top false positives (real -> scored as AIGC):


,rank,fixed_eval_index,label,source,score,file
0,0,244,0.0,sid_set,0.8533,false_positive_00_score0.853.png
1,1,240,0.0,sid_set,0.8455,false_positive_01_score0.846.png
2,2,388,0.0,wildfake,0.8000,false_positive_02_score0.800.png
3,3,416,0.0,wildfake,0.7853,false_positive_03_score0.785.png
4,4,124,0.0,sid_set,0.7836,false_positive_04_score0.784.png
5,5,11,0.0,sid_set,0.7509,false_positive_05_score0.751.png
6,6,393,0.0,wildfake,0.7481,false_positive_06_score0.748.png
7,7,3,0.0,sid_set,0.7427,false_positive_07_score0.743.png



Top false negatives (AIGC -> scored as real):


,rank,fixed_eval_index,label,source,score,file
0,0,203,1.0,sid_set,0.0090,false_negative_00_score0.009.png
1,1,527,1.0,wildfake,0.0157,false_negative_01_score0.016.png
2,2,207,1.0,sid_set,0.0791,false_negative_02_score0.079.png
3,3,44,1.0,sid_set,0.0919,false_negative_03_score0.092.png
4,4,63,1.0,sid_set,0.1341,false_negative_04_score0.134.png
5,5,428,1.0,wildfake,0.1496,false_negative_05_score0.150.png
6,6,47,1.0,sid_set,0.1509,false_negative_06_score0.151.png
7,7,519,1.0,wildfake,0.1585,false_negative_07_score0.159.png



Thumbnails + CSV saved to /content/drive/MyDrive/TechJam/checkpoints/error_analysis — inspect these and write 2-3 sentences per failure mode for the DevPost error analysis note (e.g. is there a visible pattern — heavy compression, faces, particular textures, or a specific WildFake generator underperforming vs. SID_Set?).
